# Exercise 2 — From RAG Chain to Tool-Calling Agent (Google Colab)

### Recap
In Exercise 1 you built a chain that **always** retrieves, then answers. That's great,
but a real assistant should *decide* when it needs to look something up.

### What you'll build here
You'll turn the retrieval step into a **tool** called `document_search`, then give it to
an **agent**. The agent runs a loop:

```
perceive (read the question)  →  act (maybe call document_search)  →  observe (read results)  →  answer
```

This is the **agent harness**. The magic is that *the model chooses* whether to call the
tool — you don't hard-code it.

### Prerequisites
- Completed **Exercise 1** (`story-llama` index populated)
- Colab secrets (🔑): `PINECONE_API_KEY` + **either** `GOOGLE_API_KEY` or `OPENAI_API_KEY`
- Set `LLM_PROVIDER` in Step 1 (`"google"` or `"openai"`)


## Step 0 — Install libraries
Same stack as Exercise 1, plus `langgraph`, which provides the agent loop.


In [ ]:
%pip install -q pinecone langchain langchain-core langchain-google-genai google-generativeai langchain-openai openai langgraph

## Step 1 — Load API keys & pick LLM provider
Add secrets in 🔑, then set `LLM_PROVIDER` to match the key you have.


In [ ]:
LLM_PROVIDER = "google"  # "google" | "openai"

from google.colab import userdata
import os

os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

if LLM_PROVIDER == "google":
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
elif LLM_PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    raise ValueError('LLM_PROVIDER must be "google" or "openai"')

print(f"Keys loaded · LLM provider: {LLM_PROVIDER}")

## Step 2 — Build the tool and the agent

1. **Retriever** — same `story-llama` index (Pinecone search, top `k=4`).
2. **`@tool`** — `document_search` wrapper.
3. **LLM** — Gemini or OpenAI per `LLM_PROVIDER`.
4. **`create_react_agent`** — perceive → act → observe loop.


In [ ]:
from pinecone import Pinecone
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

INDEX_NAME, NAMESPACE = "story-llama", "default"
index = Pinecone(api_key=os.environ["PINECONE_API_KEY"]).Index(INDEX_NAME)

def get_llm():
    if LLM_PROVIDER == "google":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

def pinecone_search(query: str, top_k: int = 4) -> list[str]:
    hits = index.search(namespace=NAMESPACE, query={"inputs": {"text": query}, "top_k": top_k}, fields=["chunk_text"])["result"]["hits"]
    return [h["fields"]["chunk_text"] for h in hits]

@tool
def document_search(query: str) -> str:
    """Search uploaded documents. Use when the question needs document knowledge."""
    chunks = pinecone_search(query)
    return "\n\n".join(f"Source {i+1}:\n{t}" for i, t in enumerate(chunks))

def to_text(content) -> str:
    if isinstance(content, str): return content
    if isinstance(content, dict): return content.get("text", str(content))
    if isinstance(content, list): return "".join(to_text(p) for p in content)
    return str(content)

agent = create_react_agent(model=get_llm(), tools=[document_search],
    prompt="You are an educational assistant. Answer questions about the story.")

chat_history = []

def chat(question):
    global chat_history
    result = agent.invoke({"messages": chat_history + [HumanMessage(content=question)]})
    answer = to_text(result["messages"][-1].content)
    chat_history += [HumanMessage(content=question), AIMessage(content=answer)]
    chat_history = chat_history[-8:]
    return answer

## Step 3 — Test the agent
The agent decides when to call `document_search`. Answers should be grounded in your document.


In [ ]:
questions = [
    "What is the story about?",
    "Who are the main characters?",
    "What happens at the end of the story?"
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {chat(q)}\n" + "-" * 60)

## Step 4 — Chat live
Type your own questions. Type `quit` to stop.

**Experiment:** ask something not in the document (e.g. "What's the capital of France?") and watch the agent skip the tool.


In [ ]:
while True:
    q = input("You: ").strip()
    if q.lower() in ("quit", "exit", "q"):
        break
    if q:
        print(f"Assistant: {chat(q)}\n")

## ✅ You built a tool-calling agent

| | Exercise 1 (Chain) | Exercise 2 (Agent) |
|---|---|---|
| Retrieval | **Always** runs | **Only when the agent decides** |
| Structure | Fixed pipeline | perceive → act → observe loop |

Next: Exercise 3 wraps this tool in a multi-agent team with a verifier.
